# trt_yolo_bench Colab runner

Run this notebook only in Google Colab with a GPU runtime. The first cell deliberately stops on a local CPU runtime so later cells do not produce misleading cascaded errors.

In [ ]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA runtime detected. Open this notebook in Google Colab, "
        "then choose Runtime > Change runtime type > T4 GPU and reconnect."
    )
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)
print(sys.version)

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/chao-dotcom/trt_yolo_bench.git"
REPO_DIR = Path("/content/trt_yolo_bench")
if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if not REPO_DIR.exists():
        raise RuntimeError("Repository clone failed")
os.chdir(REPO_DIR)
print(Path.cwd())

## Install dependencies

The old notebook installed `pycuda` in the same pip command as TensorRT. On current Colab Python 3.13 images, `pycuda` has no compatible wheel and that makes pip report TensorRT as missing too. This project now supports `cuda-python`, so install the packages separately.

In [ ]:
!python -m pip install -q --upgrade pip
!apt-get update -qq
!apt-get install -y -qq cmake libopencv-dev
!python -m pip install -q ultralytics onnx pycocotools requests opencv-python numpy
!python -m pip install -q tensorrt cuda-python
!python -c "import tensorrt as trt; print('TensorRT', trt.__version__)"
!python -c "from cuda import cuda; print('cuda-python import OK')"
!test -d /content/TensorRT || git clone --depth 1 -q https://github.com/NVIDIA/TensorRT.git /content/TensorRT

In [ ]:
from pathlib import Path
import os
import tensorrt_libs
headers = list(Path('/usr/local').glob('**/NvInfer.h')) + list(Path('/usr/include').glob('**/NvInfer.h'))
source_headers = Path('/content/TensorRT/include')
if not source_headers.joinpath('NvInfer.h').exists() and not headers:
    raise RuntimeError('TensorRT C++ headers are unavailable')
os.environ['TENSORRT_INCLUDE_DIR'] = str(source_headers if source_headers.joinpath('NvInfer.h').exists() else headers[0].parent)
os.environ['TENSORRT_LIB_DIR'] = str(Path(tensorrt_libs.__file__).parent)
print('TensorRT include:', os.environ['TENSORRT_INCLUDE_DIR'])
print('TensorRT libraries:', os.environ['TENSORRT_LIB_DIR'])

In [ ]:
!python export/download_coco_slice.py --val-count 1000 --calib-count 500 --insecure-downloads

In [ ]:
!python export/export_onnx.py --weights yolo11n.pt --out-dir models

In [ ]:
!python export/build_engines.py --onnx models/yolo11n.onnx --engines-dir engines --calib-cache calib.cache 2>&1 | tee build.log
from pathlib import Path
for name in ('yolo11n_fp32.engine', 'yolo11n_fp16.engine', 'yolo11n_int8.engine'):
    if not (Path('engines') / name).exists():
        raise RuntimeError(f'Missing engine after build: {name}')

In [ ]:
!cmake -S harness -B harness/build
!cmake --build harness/build --parallel
!test -x harness/build/trt_yolo_bench

In [ ]:
!./harness/build/trt_yolo_bench engines/yolo11n_fp32.engine data/val_slice.json --out results_fp32.json --latency-out latency_fp32.csv 2>&1 | tee bench_fp32.log
!./harness/build/trt_yolo_bench engines/yolo11n_fp16.engine data/val_slice.json --out results_fp16.json --latency-out latency_fp16.csv 2>&1 | tee bench_fp16.log
!./harness/build/trt_yolo_bench engines/yolo11n_int8.engine data/val_slice.json --out results_int8.json --latency-out latency_int8.csv 2>&1 | tee bench_int8.log

In [ ]:
!python eval/run_ultralytics_reference.py --weights yolo11n.pt --out results_pytorch.json
!python eval/score_coco.py --dets results_pytorch.json | tee score_pytorch.log
!python eval/score_coco.py --dets results_fp32.json | tee score_fp32.log
!python eval/score_coco.py --dets results_fp16.json | tee score_fp16.log
!python eval/score_coco.py --dets results_int8.json | tee score_int8.log

In [ ]:
!mkdir -p results
!bash -lc 'nvidia-smi > results/env.txt; nvcc --version >> results/env.txt; python -c "import tensorrt as trt; print(trt.__version__)" >> results/env.txt; pip freeze >> results/env.txt'
from google.colab import files
import shutil
shutil.make_archive('/content/trt_yolo_bench-results', 'zip', '/content/trt_yolo_bench')
files.download('/content/trt_yolo_bench-results.zip')